# 🚀 VisionAid Action Plan - Optimizations
Ce carnet implémente les deux actions recommandées pour gagner 3 à 5 points de précision (mAP/F1) sur ton modèle.

> [!IMPORTANT]
> **POUR QUE L'ACTION 1 FONCTIONNE :** Tu **DOIS** ajouter à la fois ton fichier 
> **`best.pt`** (le poids de ton modèle) **ET le dataset** dans les _Inputs Kaggle_ ! 
> Ultralytics a besoin des images de validation et de leurs labels pour calculer le F1-Score et la précision.

## 🎯 Action 1 : Trouver le Confidence Threshold Optimal
La métrique mAP balaye toutes les confiances, mais pour le **déploiement sur mobile**, tu as besoin du seuil qui maximise le **F1-Score** (l'équilibre parfait entre Precision et Recall). Nous allons balayer les seuils de `0.1` à `0.9`.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Charger ton meilleur modèle (assure-toi que best.pt existe)
try:
    model = YOLO('/kaggle/input/ton-modele-best/best.pt') # ⚠️ METS LE CHEMIN DE TON POIDS UPLOADÉ ICI
except:
    # Remplace par ton chemin réel si besoin
    model = YOLO('yolov8s.pt')

# 2. Définir la plage de test
conf_thresholds = np.arange(0.1, 0.95, 0.05)
results_list = []

print("🔍 Démarrage du balayage de Confidence...")
for conf in conf_thresholds:
    print(f"\n⏳ Test pour conf={conf:.2f}...")
    # val sur l'ensemble de validation ou test
    metrics = model.val(data='/kaggle/working/data_master.yaml', split='test', conf=conf, verbose=False)
    
    # Extraire les métriques
    p = metrics.box.mp
    r = metrics.box.mr
    map50 = metrics.box.map50
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    
    results_list.append({
        'Conf': round(conf, 2),
        'Precision': p,
        'Recall': r,
        'mAP50': map50,
        'F1-Score': f1
    })

# 3. Analyser et Trouver l'optimal
df = pd.DataFrame(results_list)
optimal_f1_idx = df['F1-Score'].idxmax()
optimal_row = df.loc[optimal_f1_idx]

print("\n" + "="*40)
print(f"🏆 SEUIL OPTIMAL TROUVÉ : {optimal_row['Conf']}")
print(f"   F1-Score Maximum : {optimal_row['F1-Score']:.4f}")
print(f"   Precision        : {optimal_row['Precision']:.4f}")
print(f"   Recall           : {optimal_row['Recall']:.4f}")
print("="*40)

# 4. Mode Visuel : Tracer les courbes
plt.figure(figsize=(10, 6))
plt.plot(df['Conf'], df['F1-Score'], marker='o', label='F1-Score', color='green')
plt.plot(df['Conf'], df['Precision'], marker='^', label='Precision', color='blue', linestyle='--')
plt.plot(df['Conf'], df['Recall'], marker='x', label='Recall', color='red', linestyle='--')
plt.axvline(x=optimal_row['Conf'], color='gold', linestyle='-', label=f'Optimal: {optimal_row["Conf"]}')
plt.title('Performance Metrics vs Confidence Threshold')
plt.xlabel('Confidence Threshold')
plt.ylabel('Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('confidence_sweep.png')
plt.show()

## 🌤️ Action 2 : Augmentation Ciblée (Luminosité, Contraste, Flou)
Pour aider le modèle sur les classes qui se fondent dans l'arrière-plan (car, person, tree, waste_container), l'augmentation photométrique est bien meilleure que la géométrique (`flip`, `crop`).
YOLOv8 dispose de ses propres hyperparamètres colorimétriques. Nous désactivons les inversions et zoom intenses, et montons le contraste/flou au maximum.

In [ ]:
from ultralytics import YOLO

# --- TEMPLATE D'ENTRAÎNEMENT OPTIMISÉ POUR ACTION 2 ---
def train_with_photometric_augmentation():
    model = YOLO('yolov8s.pt') # On part des poids COCO par défaut comme tu l'as demandé
    
    model.train(
        data='/kaggle/working/data_master.yaml',
        epochs=50,
        imgsz=832,
        batch=16,
        name='VisionAid_Action2_Augmented',
        
        # ❌ DÉSACTIVER l'augmentation de géométrique qui déforme l'obstacle :
        degrees=0.0,
        translate=0.0,
        scale=0.0,
        perspective=0.0,
        fliplr=0.0,
        flipud=0.0,
        
        # ✅ BOOSTER l'augmentation photométrique ciblée :
        hsv_h=0.015,     # Altération mineure de la Teinte
        hsv_s=0.9,       # Saturation extrême (Pousse les couleurs à se détacher du fond)
        hsv_v=0.9,       # Luminosité extrême (Contraste fort, utile pour ombres/containers)
        bgr=0.2,         # 20% de chances d'inverser RGB/BGR (Excellent pour séparer tree/background)
        blur=0.1,        # Léger flou gaussien, force le modèle à apprendre les formes générales au lieu de la texture
        clahe=0.2,       # Amélioration dynamique des contrastes (Important pour la nuit/ombre)
    )

# Décommenter la ligne suivante pour lancer l'entraînement si désiré.
# train_with_photometric_augmentation()